In [27]:
import os
import glob
import time
import numpy as np
import pandas as pd
import kagglehub
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------
# 1. Unduh Dataset via Kagglehub
# ---------------------------------------------------------------
path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")
print("Path to dataset files:", path)

# Ambil file CSV pertama sesuai instruksi soal
csv_files = sorted(glob.glob(os.path.join(path, "*.csv")))
print("Daftar file CSV ditemukan:", csv_files)
selected_csv = csv_files[0]
print("Memuat file:", selected_csv)

# ---------------------------------------------------------------
# 2. Boilerplate Modul: Load Data & Preprocessing
# ---------------------------------------------------------------
df = pd.read_csv(selected_csv)

features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Pastikan tidak ada missing value pada fitur numerik
df = df.dropna(subset=features).reset_index(drop=True)

X = df[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')

k = 10
n_samples = X_scaled.shape[0]
d = X_scaled.shape[1]
print(f"Data siap: {n_samples} baris, {d} dimensi")

# Gunakan 100 sampel pertama untuk evaluasi waktu pencarian & recall
n_queries = 100
queries = X_scaled[:n_queries]

# ---------------------------------------------------------------
# 3. Exact Nearest Neighbors (Scikit-Learn Brute Force Ground Truth)
# ---------------------------------------------------------------
print("\nMenjalankan Exact NN...")
start = time.time()
exact_nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean', n_jobs=-1)
exact_nn.fit(X_scaled)
t_build_exact = time.time() - start

start = time.time()
_, exact_indices = exact_nn.kneighbors(queries)
t_query_exact = ((time.time() - start) / n_queries) * 1000

# ---------------------------------------------------------------
# 4. ANNOY
# ---------------------------------------------------------------
print("Menjalankan ANNOY...")
annoy_index = AnnoyIndex(d, 'euclidean')

start = time.time()
for i in range(n_samples):
    annoy_index.add_item(i, X_scaled[i])
annoy_index.build(10)
t_build_annoy = time.time() - start

start = time.time()
annoy_indices = [annoy_index.get_nns_by_vector(q, k) for q in queries]
t_query_annoy = ((time.time() - start) / n_queries) * 1000
annoy_indices = np.array(annoy_indices)

# ---------------------------------------------------------------
# 5. FAISS (IndexFlatL2)
# ---------------------------------------------------------------
print("Menjalankan FAISS...")
faiss_index = faiss.IndexFlatL2(d)

start = time.time()
faiss_index.add(X_scaled)
t_build_faiss = time.time() - start

start = time.time()
_, faiss_indices = faiss_index.search(queries, k)
t_query_faiss = ((time.time() - start) / n_queries) * 1000

# ---------------------------------------------------------------
# 6. HNSWLIB
# ---------------------------------------------------------------
print("Menjalankan HNSW...")
hnsw_index = hnswlib.Index(space='l2', dim=d)

start = time.time()
hnsw_index.init_index(max_elements=n_samples, ef_construction=200, M=16)
hnsw_index.add_items(X_scaled)
t_build_hnsw = time.time() - start

hnsw_index.set_ef(50)
start = time.time()
hnsw_indices, _ = hnsw_index.knn_query(queries, k=k)
t_query_hnsw = ((time.time() - start) / n_queries) * 1000

# ---------------------------------------------------------------
# 7. Evaluasi Performa & Recall@k
# ---------------------------------------------------------------
def hitung_recall(ground_truth, pred):
    recalls = []
    for gt_row, pred_row in zip(ground_truth, pred):
        overlap = set(gt_row).intersection(set(pred_row))
        recalls.append(len(overlap) / len(gt_row))
    return np.mean(recalls) * 100

recall_exact = 100.0
recall_annoy = hitung_recall(exact_indices, annoy_indices)
recall_faiss = hitung_recall(exact_indices, faiss_indices)
recall_hnsw = hitung_recall(exact_indices, hnsw_indices)

df_evaluasi = pd.DataFrame([
    {
        "Metode": "Exact NN (Brute)",
        "Build Time (s)": f"{t_build_exact:.4f}",
        "Query Time / Item (ms)": f"{t_query_exact:.4f}",
        f"Recall@{k} (%)": f"{recall_exact:.2f}%"
    },
    {
        "Metode": "ANNOY",
        "Build Time (s)": f"{t_build_annoy:.4f}",
        "Query Time / Item (ms)": f"{t_query_annoy:.4f}",
        f"Recall@{k} (%)": f"{recall_annoy:.2f}%"
    },
    {
        "Metode": "FAISS",
        "Build Time (s)": f"{t_build_faiss:.4f}",
        "Query Time / Item (ms)": f"{t_query_faiss:.4f}",
        f"Recall@{k} (%)": f"{recall_faiss:.2f}%"
    },
    {
        "Metode": "HNSW",
        "Build Time (s)": f"{t_build_hnsw:.4f}",
        "Query Time / Item (ms)": f"{t_query_hnsw:.4f}",
        f"Recall@{k} (%)": f"{recall_hnsw:.2f}%"
    }
])

print("\n=== TABEL PERBANDINGAN PERFORMA ===")
print(df_evaluasi.to_markdown(index=False))

# ---------------------------------------------------------------
# 8. Perbandingan Hasil 10 Track Terdekat (Sampel Baris ke-0)
# ---------------------------------------------------------------
title_col = 'track_name' if 'track_name' in df.columns else df.columns[0]
artist_col = 'track_artist' if 'track_artist' in df.columns else None

def get_track_names(idx_list):
    names = []
    for idx in idx_list:
        title = df.iloc[idx][title_col]
        if artist_col:
            title += f" - {df.iloc[idx][artist_col]}"
        names.append(title)
    return names

print(f"\n=== 10 TRACK TERDEKAT UNTUK QUERY: '{df.iloc[0][title_col]}' ===")
df_lagu = pd.DataFrame({
    "Exact NN": get_track_names(exact_indices[0]),
    "ANNOY": get_track_names(annoy_indices[0]),
    "FAISS": get_track_names(faiss_indices[0]),
    "HNSW": get_track_names(hnsw_indices[0])
})
print(df_lagu.to_markdown())

100%|██████████| 894M/894M [00:45<00:00, 20.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/versions/20
Daftar file CSV ditemukan: ['/root/.cache/kagglehub/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/versions/20/songs_with_attributes_and_lyrics.csv', '/root/.cache/kagglehub/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/versions/20/songs_with_lyrics_and_timestamps.csv']
Memuat file: /root/.cache/kagglehub/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/versions/20/songs_with_attributes_and_lyrics.csv
Data siap: 955320 baris, 9 dimensi

Menjalankan Exact NN...
Menjalankan ANNOY...
Menjalankan FAISS...
Menjalankan HNSW...

=== TABEL PERBANDINGAN PERFORMA ===
| Metode           |   Build Time (s) |   Query Time / Item (ms) | Recall@10 (%)   |
|:-----------------|-----------------:|-------------------------:|:----------------|
| Exact NN (Brute) |           0.0212 |                   4.8503 | 100.00%         |
| ANNOY          